In [1]:
# 1) Install dependencies
!pip -q install -U transformers accelerate peft bitsandbytes sentencepiece huggingface_hub pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.1 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.

In [1]:
!pip install pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 41.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.1
    Uninstalling pandas-3.0.1:
      Successfully uninstalled pandas-3.0.1


In [2]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

%cd /content
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
%cd /content/llama.cpp
!cmake -B build
!cmake --build build --config Release -j 2


/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 84565, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 84565 (delta 96), reused 48 (delta 48), pack-reused 84407 (from 4)
Receiving objects: 100% (84565/84565), 332.82 MiB | 1.32 MiB/s, done.
Resolving deltas: 100% (60917/60917), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identi

In [2]:

from pathlib import Path
import os

PROJECT_ROOT = Path('/content/drive/MyDrive/Week8')
ADAPTER_DIR = PROJECT_ROOT / 'adapters'
QUANT_DIR = PROJECT_ROOT / 'quantized'
MERGED_DIR = PROJECT_ROOT / 'models' / 'merged_fp16'
REPORT_JSON = PROJECT_ROOT / 'quantized' / 'quantization_summary.json'

INT8_DIR = QUANT_DIR / 'model-int8'
INT4_DIR = QUANT_DIR / 'model-int4'
GGUF_PATH = QUANT_DIR / 'model.gguf'
GGUF_F16_PATH = QUANT_DIR / 'model-f16.gguf'

for p in [QUANT_DIR, MERGED_DIR, INT8_DIR, INT4_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('ADAPTER_DIR :', ADAPTER_DIR)
print('MERGED_DIR  :', MERGED_DIR)
print('INT8_DIR    :', INT8_DIR)
print('INT4_DIR    :', INT4_DIR)
print('GGUF_PATH   :', GGUF_PATH)


ADAPTER_DIR : /content/drive/MyDrive/Week8/adapters
MERGED_DIR  : /content/drive/MyDrive/Week8/models/merged_fp16
INT8_DIR    : /content/drive/MyDrive/Week8/quantized/model-int8
INT4_DIR    : /content/drive/MyDrive/Week8/quantized/model-int4
GGUF_PATH   : /content/drive/MyDrive/Week8/quantized/model.gguf


In [3]:

required = [
    ADAPTER_DIR / 'adapter_model.bin',
    ADAPTER_DIR / 'adapter_config.json',
]
for p in required:
    print(p, '->', p.exists())


/content/drive/MyDrive/Week8/adapters/adapter_model.bin -> True
/content/drive/MyDrive/Week8/adapters/adapter_config.json -> True


In [6]:
# 6) Imports and config
import json
import time
import shutil
import subprocess
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

BASE_MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
TEST_PROMPTS = [
    "### Instruction:\nAnswer this medical question.\n\n### Input:\nWhat is pneumonia?\n\n### Response:\n",
    "### Instruction:\nReason briefly and answer.\n\n### Input:\nA patient has unilateral leg swelling after a long flight and sudden neurological deficits. What cardiac finding may explain this?\n\n### Response:\n",
]

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [7]:
# 7) Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
print('Tokenizer loaded.')


Tokenizer loaded.


In [8]:
# 8) Load base FP16 model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)

peft_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
merged_model = peft_model.merge_and_unload()

merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))

print('Merged model saved to:', MERGED_DIR)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/drive/MyDrive/Week8/models/merged_fp16


In [9]:
# 9) Load and save 8-bit model
bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    quantization_config=bnb_int8,
    device_map='auto',
)

model_int8.save_pretrained(str(INT8_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(INT8_DIR))

print('Saved INT8 model to:', INT8_DIR)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved INT8 model to: /content/drive/MyDrive/Week8/quantized/model-int8


In [10]:
# 10) Load and save 4-bit model
bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    quantization_config=bnb_int4,
    device_map='auto',
)

model_int4.save_pretrained(str(INT4_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(INT4_DIR))

print('Saved INT4 model to:', INT4_DIR)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved INT4 model to: /content/drive/MyDrive/Week8/quantized/model-int4


In [11]:
# 11) Convert merged model to GGUF (F16 first, then quantize to q4_0)
%cd /content/llama.cpp

!python convert_hf_to_gguf.py "{MERGED_DIR}" --outfile "{GGUF_F16_PATH}" --outtype f16
!./build/bin/llama-quantize "{GGUF_F16_PATH}" "{GGUF_PATH}" q4_0

print('Saved GGUF model to:', GGUF_PATH)


/content/llama.cpp
INFO:hf-to-gguf:Loading model: merged_fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:bl

In [12]:
# 12) Utility: get directory/file sizes
def get_size_mb(path: Path) -> float:
    if path.is_file():
        return path.stat().st_size / (1024 ** 2)
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024 ** 2)

sizes = {
    'FP16_merged_mb': get_size_mb(MERGED_DIR),
    'INT8_mb': get_size_mb(INT8_DIR),
    'INT4_mb': get_size_mb(INT4_DIR),
    'GGUF_q4_0_mb': get_size_mb(GGUF_PATH),
}
sizes


{'FP16_merged_mb': 2101.6503896713257,
 'INT8_mb': 1179.185625076294,
 'INT4_mb': 730.5854063034058,
 'GGUF_q4_0_mb': 607.2297973632812}

In [13]:
# 13) Quick HF generation benchmark helper
def benchmark_hf_model(model, tokenizer, prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    num_tokens = generated.shape[0]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return {
        'seconds': round(elapsed, 4),
        'new_tokens': int(num_tokens),
        'tokens_per_sec': round(num_tokens / max(elapsed, 1e-6), 4),
        'text': text,
    }


In [14]:
# 14) Run quick quality + speed check for FP16 / INT8 / INT4
records = []

# reload small references cleanly
model_fp16 = AutoModelForCausalLM.from_pretrained(str(MERGED_DIR), torch_dtype=torch.float16, device_map='auto')
model_int8_eval = AutoModelForCausalLM.from_pretrained(str(INT8_DIR), device_map='auto')
model_int4_eval = AutoModelForCausalLM.from_pretrained(str(INT4_DIR), device_map='auto')

for prompt in TEST_PROMPTS:
    fp16_result = benchmark_hf_model(model_fp16, tokenizer, prompt)
    int8_result = benchmark_hf_model(model_int8_eval, tokenizer, prompt)
    int4_result = benchmark_hf_model(model_int4_eval, tokenizer, prompt)

    records.extend([
        {'format': 'FP16', 'prompt': prompt, **fp16_result},
        {'format': 'INT8', 'prompt': prompt, **int8_result},
        {'format': 'INT4', 'prompt': prompt, **int4_result},
    ])

hf_df = pd.DataFrame(records)
hf_df.head()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Pl

,format,prompt,seconds,new_tokens,tokens_per_sec,text
0,FP16,Explain the difference between bacterial and v...,3.5599,80,22.4723,\nSolve the medical reasoning problem carefull...
1,INT8,Explain the difference between bacterial and v...,8.9044,80,8.9843,\nSolve the medical reasoning problem carefull...
2,INT4,Explain the difference between bacterial and v...,0.1128,1,8.8674,
3,FP16,What is the likely diagnosis for sudden unilat...,2.5269,80,31.6597,\nAnswer: What is the likely diagnosis for sud...
4,INT8,What is the likely diagnosis for sudden unilat...,7.9301,80,10.0882,\nAnswer : What is the likely diagnosis for su...


In [9]:
from pathlib import Path
print(GGUF_PATH)
print(Path(GGUF_PATH).exists())
print(Path(GGUF_PATH).stat().st_size / (1024**3), "GB")

/content/drive/MyDrive/Week8/quantized/model.gguf
True
0.5929978489875793 GB


In [2]:
from pathlib import Path

quant_dir = Path("/content/drive/MyDrive/Week8/quantized")

paths = {
    "int8_dir": quant_dir / "model-int8",
    "int4_dir": quant_dir / "model-int4",
    "gguf_file": quant_dir / "model.gguf",
}

for name, path in paths.items():
    print(name, "->", path.exists(), path)
    if path.exists() and path.is_file():
        print("  size_gb:", round(path.stat().st_size / (1024**3), 4))

int8_dir -> True /content/drive/MyDrive/Week8/quantized/model-int8
int4_dir -> True /content/drive/MyDrive/Week8/quantized/model-int4
gguf_file -> True /content/drive/MyDrive/Week8/quantized/model.gguf
  size_gb: 0.593


In [3]:
import os
import json
import pandas as pd
from pathlib import Path

quant_dir = Path("/content/drive/MyDrive/Week8/quantized")
records = []

items = [
    ("INT8", quant_dir / "model-int8"),
    ("INT4", quant_dir / "model-int4"),
    ("GGUF_q4_0", quant_dir / "model.gguf"),
]

for label, path in items:
    if path.exists():
        if path.is_file():
            size_bytes = path.stat().st_size
        else:
            size_bytes = 0
            for root, _, files in os.walk(path):
                for f in files:
                    fp = Path(root) / f
                    if fp.exists():
                        size_bytes += fp.stat().st_size

        records.append({
            "format": label,
            "path": str(path),
            "exists": True,
            "size_mb": round(size_bytes / (1024**2), 2),
            "size_gb": round(size_bytes / (1024**3), 4),
            "status": "generated"
        })
    else:
        records.append({
            "format": label,
            "path": str(path),
            "exists": False,
            "size_mb": None,
            "size_gb": None,
            "status": "missing"
        })

df = pd.DataFrame(records)
df

,format,path,exists,size_mb,size_gb,status
0,INT8,/content/drive/MyDrive/Week8/quantized/model-int8,True,1179.19,1.1515,generated
1,INT4,/content/drive/MyDrive/Week8/quantized/model-int4,True,730.59,0.7135,generated
2,GGUF_q4_0,/content/drive/MyDrive/Week8/quantized/model.gguf,True,607.23,0.5930,generated


In [4]:
out_csv = quant_dir / "quantization_summary.csv"
out_json = quant_dir / "quantization_summary.json"

df.to_csv(out_csv, index=False)
df.to_json(out_json, orient="records", indent=2)

print("Saved:", out_csv)
print("Saved:", out_json)

Saved: /content/drive/MyDrive/Week8/quantized/quantization_summary.csv
Saved: /content/drive/MyDrive/Week8/quantized/quantization_summary.json
